# Computational Trade-offs and Batching Benchmark

This notebook validates the claims made in the `Computational Trade-offs and Batching` section of the manuscript.
It provides a comprehensive measurement of inference speed and memory usage for **Prompt U-Net (v330/v332 architecture)**, **UniverSeg**, and **nnInteractive**.

**Settings evaluated:**
- CPU Execution (Single Batch) to demonstrate memory efficiency in resource-constrained environments.
- GPU Execution (Max Batch) to demonstrate throughput improvements via parallel compute saturation.

**Structures evaluated (40 2D slices each):**
- **Small (128x128):** Fits within the native tile size, zero tiling overhead for Prompt U-Net.
- **Medium (256x256):** Requires adaptive tiling for Prompt U-Net.
- **Large (512x512):** Requires extensive adaptive tiling for Prompt U-Net.

*Note: This benchmark uses an isolated subprocess pattern. Each model runs in a separate process to guarantee that TensorFlow and PyTorch memory allocations do not conflict and that GPU memory is perfectly reset between evaluations.*


In [ ]:
import os
import sys
import json
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12})

SIZES = {
    "Small (128x128)": (128, 128),
    "Medium (256x256)": (256, 256),
    "Large (512x512)": (512, 512)
}

ENVIRONMENTS = ["CPU (Batch=1)", "GPU (Max Batch)"]
MODELS = ["Prompt U-Net", "UniverSeg", "nnInteractive"]


In [ ]:
worker_code = """import sys
import os
import gc
import time
import json
import numpy as np
import threading
import psutil

# Setup paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

class MemoryMonitor:
    def __init__(self):
        self.keep_measuring = True
        self.peak_memory = 0.0
        
    def measure(self):
        p = psutil.Process(os.getpid())
        while self.keep_measuring:
            try:
                mem = p.memory_info().rss / (1024**2)
                if mem > self.peak_memory:
                    self.peak_memory = mem
                time.sleep(0.005)
            except:
                break

def clean_memory():
    gc.collect()
    if 'torch' in sys.modules:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
    if 'tensorflow' in sys.modules:
        import tensorflow as tf
        tf.keras.backend.clear_session()
        try:
            tf.config.experimental.reset_memory_stats('GPU:0')
        except:
            pass

def main():
    model_name = sys.argv[1]
    env = sys.argv[2]
    H = int(sys.argv[3])
    W = int(sys.argv[4])
    
    GPU_BATCH_SIZE_PUNET = 32
    GPU_BATCH_SIZE_UNIVERSEG = 32
    NUM_SLICES = 40
    
    clean_memory()
    
    if model_name == "Prompt U-Net":
        import tensorflow as tf
        # Set TF memory growth
        gpus = tf.config.experimental.list_physical_devices('GPU')
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
            
        from inference.predictor import PromptUNetPredictor
        punet_path = os.path.join(PROJECT_ROOT, "training", "p_unet_330.keras")
        punet_predictor = PromptUNetPredictor(punet_path)
        
        device_name = '/CPU:0' if 'CPU' in env else '/GPU:0'
        batch_size = 1 if 'CPU' in env else GPU_BATCH_SIZE_PUNET
        
        x = np.random.randn(NUM_SLICES, H, W).astype(np.float32)
        p = np.zeros((NUM_SLICES, H, W, 2), dtype=np.float32)
        
        y_start, y_end = int(H * 0.1), int(H * 0.9)
        x_start, x_end = int(W * 0.1), int(W * 0.9)
        p[:, y_start:y_end, x_start:x_end, 1] = 1.0 
        
        clean_memory()
        
        with tf.device(device_name):
            _ = punet_predictor.predict(x, p, batch_size=batch_size)
            
        clean_memory()
        
        baseline_mem = psutil.Process(os.getpid()).memory_info().rss / (1024**2)
        if 'CPU' in env:
            monitor = MemoryMonitor()
            t_thread = threading.Thread(target=monitor.measure)
            t_thread.start()
            
        start_time = time.perf_counter()
        with tf.device(device_name):
            _ = punet_predictor.predict(x, p, batch_size=batch_size)
        end_time = time.perf_counter()
        
        if 'CPU' in env:
            monitor.keep_measuring = False
            t_thread.join()
            mem_mb = max(0.0, monitor.peak_memory - baseline_mem)
        else:
            try:
                mem_info = tf.config.experimental.get_memory_info('GPU:0')
                mem_mb = mem_info['peak'] / (1024**2)
            except:
                mem_mb = 0.0
            
        return (end_time - start_time) * 1000, mem_mb

    elif model_name == "UniverSeg":
        import torch
        from evaluation.benchmark_models.UniverSeg.universeg import universeg
        
        device = torch.device('cpu' if 'CPU' in env else 'cuda:0')
        batch_size = 1 if 'CPU' in env else GPU_BATCH_SIZE_UNIVERSEG
        
        universeg_model = universeg(pretrained=False)
        universeg_model.eval()
        model = universeg_model.to(device)
        
        x = torch.randn(NUM_SLICES, 1, H, W)
        sup_x = torch.randn(NUM_SLICES, 16, 1, 128, 128)
        sup_y = torch.zeros(NUM_SLICES, 16, 1, 128, 128)
        
        clean_memory()
        
        with torch.no_grad():
            x_128 = torch.nn.functional.interpolate(x[:1].to(device), size=(128, 128), mode='bilinear', align_corners=False)
            _ = model(x_128, sup_x[:1].to(device), sup_y[:1].to(device))
            if 'GPU' in env: torch.cuda.synchronize()
            
        clean_memory()
        
        baseline_mem = psutil.Process(os.getpid()).memory_info().rss / (1024**2)
        if 'CPU' in env:
            monitor = MemoryMonitor()
            t_thread = threading.Thread(target=monitor.measure)
            t_thread.start()
            
        start_time = time.perf_counter()
        with torch.no_grad():
            for i in range(0, NUM_SLICES, batch_size):
                batch_x = x[i:i+batch_size].to(device)
                batch_sx = sup_x[i:i+batch_size].to(device)
                batch_sy = sup_y[i:i+batch_size].to(device)
                
                batch_x_128 = torch.nn.functional.interpolate(batch_x, size=(128, 128), mode='bilinear', align_corners=False)
                logits = model(batch_x_128, batch_sx, batch_sy)
                _ = torch.nn.functional.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)
                
        if 'GPU' in env: torch.cuda.synchronize()
        end_time = time.perf_counter()
        
        if 'CPU' in env:
            monitor.keep_measuring = False
            t_thread.join()
            mem_mb = max(0.0, monitor.peak_memory - baseline_mem)
        else:
            mem_mb = torch.cuda.max_memory_allocated(device) / (1024**2) if device.type == 'cuda' else 0.0
        return (end_time - start_time) * 1000, mem_mb

    elif model_name == "nnInteractive":
        import torch
        from huggingface_hub import snapshot_download
        from nnInteractive.inference.inference_session import nnInteractiveInferenceSession
        
        device = torch.device('cpu' if 'CPU' in env else 'cuda:0')
        
        model_path = snapshot_download(
            repo_id="nnInteractive/nnInteractive",
            allow_patterns=["nnInteractive_v1.0/*"],
            local_dir="./nninteractive_model"
        )
        nn_session = nnInteractiveInferenceSession(device=torch.device('cpu'), use_torch_compile=False, verbose=False)
        nn_session.initialize_from_trained_model_folder(os.path.join(model_path, "nnInteractive_v1.0"))
        nninteractive_model = nn_session.network
        nninteractive_model.eval()
        model = nninteractive_model.to(device)
        
        C_IN = 8
        
        # 3D UNet downsamples 5 times, meaning dimensions must be divisible by 2^5 = 32.
        # num_slices=40 is not divisible by 32, so we must pad the depth to avoid shape mismatches.
        pad_d = ((NUM_SLICES + 31) // 32) * 32 - NUM_SLICES
        dummy_input = torch.randn(1, C_IN, NUM_SLICES + pad_d, H, W).to(device)
        
        clean_memory()
        
        try:
            with torch.no_grad():
                _ = model(dummy_input)
                if 'GPU' in env: torch.cuda.synchronize()
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                return float('nan'), float('nan')
            raise e
            
        clean_memory()
        
        baseline_mem = psutil.Process(os.getpid()).memory_info().rss / (1024**2)
        if 'CPU' in env:
            monitor = MemoryMonitor()
            t_thread = threading.Thread(target=monitor.measure)
            t_thread.start()
            
        start_time = time.perf_counter()
        try:
            with torch.no_grad():
                _ = model(dummy_input)
                if 'GPU' in env: torch.cuda.synchronize()
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                return float('nan'), float('nan')
            raise e
            
        end_time = time.perf_counter()
        
        if 'CPU' in env:
            monitor.keep_measuring = False
            t_thread.join()
            mem_mb = max(0.0, monitor.peak_memory - baseline_mem)
        else:
            mem_mb = torch.cuda.max_memory_allocated(device) / (1024**2) if device.type == 'cuda' else 0.0
        return (end_time - start_time) * 1000, mem_mb

if __name__ == "__main__":
    try:
        t, mem = main()
        print(json.dumps({"time": t, "memory": mem}))
    except Exception as e:
        print(json.dumps({"error": str(e)}))
"""

with open("benchmark_worker.py", "w", encoding="utf-8") as f:
    f.write(worker_code)
print("Worker script written to benchmark_worker.py")


In [ ]:
# Run Benchmarks
results = []

for env in ENVIRONMENTS:
    print(f"\n--- Evaluating on {env} ---")
    for size_name, (H, W) in SIZES.items():
        print(f"  Structure Size: {size_name}")
        
        for model in MODELS:
            # Execute worker process
            cmd = [sys.executable, "benchmark_worker.py", model, env, str(H), str(W)]
            proc = subprocess.run(cmd, capture_output=True, text=True)
            
            if proc.returncode != 0:
                print(f"    {model:<13} : Failed | {proc.stderr.strip()[:100]}")
                results.append({"Environment": env, "Size": size_name, "Model": model, "Time (ms)": np.nan, "Memory (MB)": np.nan})
                continue
                
            try:
                out_lines = [l for l in proc.stdout.split("\n") if l.strip().startswith("{") and "time" in l]
                if out_lines:
                    res = json.loads(out_lines[-1])
                    if "error" in res:
                        print(f"    {model:<13} : Error | {res['error'][:100]}")
                        t, mem = np.nan, np.nan
                    else:
                        t, mem = res["time"], res["memory"]
                else:
                    print(f"    {model:<13} : Parse Err | {proc.stdout.strip()[:100]}")
                    t, mem = np.nan, np.nan
            except Exception as e:
                print(f"    {model:<13} : Exception | {e}")
                t, mem = np.nan, np.nan
                
            results.append({"Environment": env, "Size": size_name, "Model": model, "Time (ms)": t, "Memory (MB)": mem})
            if np.isnan(t):
                pass # Error message already printed
            else:
                print(f"    {model:<13} : {t:.1f} ms, {mem:.1f} MB")

df = pd.DataFrame(results)
display(df)

# Cleanup
if os.path.exists("benchmark_worker.py"):
    os.remove("benchmark_worker.py")


In [ ]:
# Visualization

# Filter out OOM (NaN) for plotting
df_plot = df.dropna()

# Inference Time Plot
g = sns.catplot(
    data=df, kind="bar",
    x="Size", y="Time (ms)", hue="Model", col="Environment",
    palette=["#4c72b0", "#dd8452", "#55a868"],
    height=6, aspect=1.2, sharey=False
)
g.fig.suptitle("Volume Inference Time (40 slices) - Lower is Better", y=1.05, fontweight="bold")
for ax in g.axes.flat:
    ax.set_yscale("log")
    ax.set_ylabel("Time (ms) [Log Scale]")
    
    # Add annotations
    for p in ax.patches:
        height = p.get_height()
        if not np.isnan(height) and height > 0:
            ax.annotate(f'{height:.0f}', (p.get_x() + p.get_width() / 2., height),
                        ha='center', va='bottom', fontsize=10, rotation=90, xytext=(0, 5),
                        textcoords='offset points')

plt.show()

# Memory Plot (Only valid for GPU)
df_gpu = df[df["Environment"] == "GPU (Max Batch)"].copy()
plt.figure(figsize=(8, 6))
sns.barplot(data=df_gpu, x="Size", y="Memory (MB)", hue="Model", palette=["#4c72b0", "#dd8452", "#55a868"])
plt.title("Peak GPU Memory Usage", fontweight="bold")
plt.ylabel("Memory (MB)")

# Add annotations
ax = plt.gca()
for p in ax.patches:
    height = p.get_height()
    if not np.isnan(height) and height > 0:
        ax.annotate(f'{height:.0f}', (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom', fontsize=10)

plt.show()
